# New Figures Phase (A-E)

Implements the five new visualisation figures defined in the plan.
All figures read from `data/processed/q1_history_base_dref.csv` produced by `01_data_prep_and_contract.ipynb`.
Outputs are written to `outputs/new_figures/`.

| Figure | Description |
|--------|-------------|
| A | Regional allocation evolution — stacked area chart |
| B | Disaster-type ranking + year-over-year trend panel |
| C | Pillar growth over time — CHF and operation count |
| D | Timeliness improvement slope chart (Q1 2022-2026) |
| E | Regional allocation heatmap (CHF and operations) |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'new_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
q1_history = pd.read_csv(
    PROCESSED_DIR / 'q1_history_base_dref.csv',
    parse_dates=['approval_date', 'date_of_appeal_request_from_ns']
)
q1_history['approval_year'] = pd.to_numeric(q1_history['approval_year'], errors='coerce').astype('Int64')
q1_history['total_approved_chf'] = pd.to_numeric(q1_history['total_approved_chf'], errors='coerce').fillna(0)
q1_history['average_time_disaster_to_approval'] = pd.to_numeric(q1_history['average_time_disaster_to_approval'], errors='coerce')
q1_history['days_in_hq'] = pd.to_numeric(q1_history['days_in_hq'], errors='coerce')

# Flourish-inspired style system
FLR_PAPER = '#fafafa'
FLR_PLOT = '#fafafa'
FLR_GRID = '#e8e8e8'
FLR_LINE = '#d0d0d0'

FLR_FONT = 'Inter, Segoe UI, Helvetica Neue, Arial, sans-serif'
FLR_TITLE = {'size': 30, 'color': '#1a1a2e', 'family': f'Inter, Segoe UI Semibold, {FLR_FONT}'}
FLR_SUBTITLE = {'size': 18, 'color': '#666666', 'family': FLR_FONT}
FLR_AXIS_LBL = {'size': 20, 'color': '#333333', 'family': FLR_FONT}
FLR_TICK = {'size': 17, 'color': '#555555', 'family': FLR_FONT}


def flr_legend(y=-0.24, x=0.5):
    return dict(
        orientation='h',
        xanchor='center',
        x=x,
        yanchor='top',
        y=y,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font={'size': 15, 'color': '#555555', 'family': FLR_FONT},
        itemsizing='constant',
    )


def flr_xaxis(title='', **overrides):
    base = dict(
        title={'text': title, 'font': FLR_AXIS_LBL, 'standoff': 16},
        tickfont=FLR_TICK,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=FLR_LINE,
        linewidth=1,
        tickcolor=FLR_LINE,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        automargin=True,
    )
    base.update(overrides)
    return base


def flr_yaxis(title='', **overrides):
    base = dict(
        title={'text': title, 'font': FLR_AXIS_LBL, 'standoff': 16},
        tickfont=FLR_TICK,
        gridcolor=FLR_GRID,
        gridwidth=0.7,
        showgrid=True,
        zeroline=False,
        showline=False,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        automargin=True,
    )
    base.update(overrides)
    return base


def flr_layout(title_text, subtitle='', height=700, margin_l=90, margin_r=50, margin_t=125, margin_b=150):
    title_html = title_text
    if subtitle:
        title_html += f'<br><span style="font-size:18px;color:#888;">{subtitle}</span>'
    return dict(
        title={
            'text': title_html,
            'x': 0.5,
            'xanchor': 'center',
            'font': FLR_TITLE,
            'pad': {'b': 18},
        },
        paper_bgcolor=FLR_PAPER,
        plot_bgcolor=FLR_PLOT,
        margin={'l': margin_l, 'r': margin_r, 't': margin_t, 'b': margin_b},
        height=height,
        font={'family': FLR_FONT, 'size': 15, 'color': '#444444'},
        hoverlabel=dict(
            bgcolor='white',
            bordercolor='#ddd',
            font_size=14,
            font_family=FLR_FONT,
        ),
    )


IFRC_RED = '#E03C31'
NAVY = '#2B3A67'
AMBER = '#F5A623'
WARM_ORANGE = '#F08A4B'
TEAL = '#2EC4B6'
GREY = '#D5D5D5'
INK = '#222222'
BG = '#fafafa'

REGION_COLORS = {
    'Africa': '#E8733A',
    'Americas': '#3DAD6E',
    'Asia-Pacific': '#4ECDC4',
    'Europe': '#9B72CF',
    'MENA': '#4A90D9',
    'Global': '#B0B0B0',
}

HAZARD_PALETTE = ['#E03C31', '#F08A4B', '#F5C518', '#3A9BD5', '#3DAD6E']


def human_chf(value):
    if pd.isna(value):
        return 'n/a'
    if abs(value) >= 1_000_000:
        return f'CHF {value / 1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'CHF {value / 1_000:.0f}K'
    return f'CHF {value:,.0f}'


def save_figure(fig, stem, width=1400, height=900):
    fig.write_image(OUTPUT_DIR / f'{stem}.png', width=width, height=height, scale=2)
    fig.write_image(OUTPUT_DIR / f'{stem}.svg', width=width, height=height)


print('Q1 history rows:', len(q1_history))
print('Years:', sorted(q1_history['approval_year'].dropna().unique().tolist()))
print('Output dir:', OUTPUT_DIR)

Q1 history rows: 180
Years: [2022, 2023, 2024, 2025, 2026]
Output dir: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\new_figures


## Figure A: Regional Allocation Evolution

Stacked area chart showing how approved CHF breaks down by region across each Q1 from 2022 to 2026.
Regions are stacked in descending order of total CHF so the dominant region sits at the base.

In [3]:
# Figure A: Regional Allocation Evolution
area_data = (
    q1_history
    .groupby(['approval_year', 'region'], as_index=False)['total_approved_chf']
    .sum()
)
area_data['approval_year'] = area_data['approval_year'].astype(int)

region_order = (
    area_data.groupby('region')['total_approved_chf']
    .sum().sort_values(ascending=False).index.tolist()
)

total_2026 = area_data[area_data['approval_year'] == 2026]['total_approved_chf'].sum()
top_region_2026 = (
    area_data[area_data['approval_year'] == 2026]
    .sort_values('total_approved_chf', ascending=False)
    .iloc[0]
)
top_region_share = top_region_2026['total_approved_chf'] / total_2026

fig = go.Figure()
for region in region_order:
    subset = area_data[area_data['region'] == region].sort_values('approval_year')
    color = REGION_COLORS.get(region, GREY)
    fig.add_trace(go.Scatter(
        x=subset['approval_year'],
        y=subset['total_approved_chf'],
        mode='lines',
        name=region,
        stackgroup='one',
        fillcolor=color,
        line={'color': 'rgba(255,255,255,0.7)', 'width': 1.3},
        hovertemplate=f'<b>{region}</b><br>Q1 %{{x}}: CHF %{{y:,.0f}}<extra></extra>',
    ))

fig.add_annotation(
    x=2026,
    y=total_2026 * 0.95,
    xref='x',
    yref='y',
    text=(f'<b>{human_chf(total_2026)}</b> — Q1 record<br>'
          f'<span style="font-size:13px;color:#888;">{top_region_2026["region"]} drives {top_region_share:.0%}</span>'),
    showarrow=True,
    ax=-80,
    ay=-45,
    arrowcolor=INK,
    arrowwidth=1.5,
    arrowhead=2,
    bgcolor='white',
    bordercolor='#ddd',
    borderwidth=1,
    borderpad=8,
    font={'size': 14, 'color': INK, 'family': FLR_FONT},
    align='left'
)

fig.update_layout(
    **flr_layout(
        'Regional Allocation Evolution',
        subtitle='Approved CHF by region — Q1 2022 to 2026',
        height=690,
        margin_b=165,
        margin_t=125,
        margin_r=70,
    ),
    legend=flr_legend(y=-0.18),
    xaxis=flr_xaxis('', tickmode='linear', dtick=1),
    yaxis=flr_yaxis('Approved CHF', tickformat=',.0s'),
)

save_figure(fig, 'figure_a_regional_evolution', width=1400, height=690)
fig

## Figures B1 & B2: Disaster-Type Ranking + Year-over-Year Trend

Split into two standalone figures:
- **B1** — 2026 Q1 ranking: top 5 hazards by approved CHF (horizontal bar)
- **B2** — Q1 trend 2022–2026: multi-year line chart for those same 5 hazards

In [4]:
# ── Shared data prep for B1 and B2 ───────────────────────────────────────────
hazard_all = (
    q1_history
    .groupby('disaster_definition', as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'))
)
top5 = hazard_all.nlargest(5, 'total_chf')['disaster_definition'].tolist()
hazard_color_map = {h: HAZARD_PALETTE[i] for i, h in enumerate(top5)}

# 2026 Q1 snapshot with ops count
snap_2026 = (
    q1_history[q1_history['approval_year'] == 2026]
    .groupby('disaster_definition', as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'), ops=('appeal_id', 'nunique'))
)
snap_2026 = snap_2026[snap_2026['disaster_definition'].isin(top5)].sort_values('total_chf')

# Multi-year trend
hazard_trend = (
    q1_history[q1_history['disaster_definition'].isin(top5)]
    .groupby(['approval_year', 'disaster_definition'], as_index=False)['total_approved_chf']
    .sum()
)
hazard_trend['approval_year'] = hazard_trend['approval_year'].astype(int)

# ── Figure B1: 2026 Q1 Ranking with operation count badges ──────────────────
bar_colors = [hazard_color_map.get(h, GREY) for h in snap_2026['disaster_definition']]

fig_b1 = go.Figure()
fig_b1.add_trace(go.Bar(
    x=snap_2026['total_chf'],
    y=snap_2026['disaster_definition'],
    orientation='h',
    marker_color=bar_colors,
    marker_line_width=0,
    marker_cornerradius=4,
    text=[f'{human_chf(v)}  │  {ops} ops' for v, ops in zip(snap_2026['total_chf'], snap_2026['ops'])],
    textposition='outside',
    textfont={'size': 13, 'color': INK, 'family': FLR_FONT},
    cliponaxis=False,
    showlegend=False,
))

fig_b1.update_layout(
    **flr_layout(
        'Top 5 Hazards — Q1 2026 Ranking',
        subtitle='Approved CHF and operation count per disaster type',
        height=520, margin_b=60, margin_t=100, margin_l=160, margin_r=180,
    ),
    xaxis=flr_xaxis('Approved CHF', range=[0, snap_2026['total_chf'].max() * 1.40]),
    yaxis=flr_yaxis('', showgrid=False),
)
fig_b1.update_yaxes(tickfont={'size': 14, 'color': INK, 'family': FLR_FONT})

save_figure(fig_b1, 'figure_b1_hazard_ranking_2026', width=1200, height=520)
fig_b1

In [5]:
# Figure B2: Q1 Trend 2022–2026
fig_b2 = go.Figure()
for i, hazard in enumerate(top5):
    subset = hazard_trend[hazard_trend['disaster_definition'] == hazard].sort_values('approval_year')
    fig_b2.add_trace(go.Scatter(
        x=subset['approval_year'],
        y=subset['total_approved_chf'],
        mode='lines+markers',
        name=hazard,
        line={'color': HAZARD_PALETTE[i], 'width': 3, 'shape': 'linear'},
        marker={'size': 11, 'color': HAZARD_PALETTE[i], 'line': {'color': FLR_PAPER, 'width': 1.5}},
        hovertemplate=f'<b>{hazard}</b><br>Q1 %{{x}}: CHF %{{y:,.0f}}<extra></extra>',
    ))

fig_b2.update_layout(
    **flr_layout(
        'Top 5 Hazards — Q1 Trend',
        subtitle='Approved CHF, 2022 to 2026 (linear interpolation — data points are annual)',
        height=610,
        margin_b=175,
        margin_t=125,
    ),
    legend=flr_legend(y=-0.20),
    xaxis=flr_xaxis('', tickmode='linear', dtick=1),
    yaxis=flr_yaxis('Approved CHF', tickformat=',.0s'),
)

save_figure(fig_b2, 'figure_b2_hazard_trend', width=1200, height=610)
fig_b2

## Figure C1 & C2: Pillar Growth Over Time

Split into two standalone figures so titles never overlap legends:
- **C1** — Approved CHF stacked bars by pillar, Q1 2022–2026
- **C2** — Operation count lines by pillar, Q1 2022–2026

In [6]:
# Figure C1: Approved CHF by Pillar
pillar_data = (
    q1_history
    .groupby(['approval_year', 'pillar'], as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'), ops=('appeal_id', 'nunique'))
)
pillar_data['approval_year'] = pillar_data['approval_year'].astype(int)

PILLAR_COLORS = {'Response': IFRC_RED, 'Anticipatory': AMBER}
year_totals = pillar_data.groupby('approval_year')['total_chf'].sum()

fig_c1 = go.Figure()
for pillar, color in [('Response', IFRC_RED), ('Anticipatory', AMBER)]:
    subset = pillar_data[pillar_data['pillar'] == pillar].sort_values('approval_year')
    text_pos = []
    for _, row in subset.iterrows():
        year_total = year_totals.get(row['approval_year'], 1)
        share = row['total_chf'] / year_total if year_total > 0 else 0
        text_pos.append('inside' if share > 0.08 else 'none')

    fig_c1.add_trace(go.Bar(
        x=subset['approval_year'],
        y=subset['total_chf'],
        name=pillar,
        marker_color=color,
        marker_line_width=0,
        marker_cornerradius=4,
        text=[human_chf(value) for value in subset['total_chf']],
        textposition=text_pos,
        textfont={'color': 'white', 'size': 14, 'family': FLR_FONT},
    ))

    for _, row in subset.iterrows():
        year_total = year_totals.get(row['approval_year'], 1)
        share = row['total_chf'] / year_total if year_total > 0 else 0
        if share <= 0.08 and row['total_chf'] > 0:
            fig_c1.add_annotation(
                x=row['approval_year'],
                y=year_total,
                text=f'<span style="font-size:13px;color:{color};">{human_chf(row["total_chf"])}</span>',
                showarrow=False,
                yshift=16,
                font={'family': FLR_FONT},
            )

fig_c1.update_layout(
    **flr_layout(
        'Approved CHF by Pillar',
        subtitle='Response vs Anticipatory — Q1 2022 to 2026',
        height=590,
        margin_b=165,
        margin_t=125,
    ),
    barmode='stack',
    legend=flr_legend(y=-0.18),
    xaxis=flr_xaxis('', tickmode='linear', dtick=1),
    yaxis=flr_yaxis('Approved CHF', tickformat=',.0s'),
)

save_figure(fig_c1, 'figure_c1_pillar_chf', width=1200, height=590)
fig_c1

In [7]:
# Figure C2: Operations by Pillar
fig_c2 = go.Figure()
for pillar, color in [('Response', IFRC_RED), ('Anticipatory', AMBER)]:
    subset = pillar_data[pillar_data['pillar'] == pillar].sort_values('approval_year')
    fig_c2.add_trace(go.Scatter(
        x=subset['approval_year'],
        y=subset['ops'],
        mode='lines+markers+text',
        name=pillar,
        line={'color': color, 'width': 3, 'shape': 'linear'},
        marker={'size': 11, 'color': color, 'line': {'color': FLR_PAPER, 'width': 1.5}},
        text=[str(int(value)) for value in subset['ops']],
        textposition='top center',
        textfont={'color': color, 'size': 15, 'family': FLR_FONT},
    ))

fig_c2.update_layout(
    **flr_layout(
        'Operations by Pillar',
        subtitle='Number of DREF operations — Q1 2022 to 2026',
        height=520,
        margin_b=165,
        margin_t=125,
    ),
    legend=flr_legend(y=-0.18),
    xaxis=flr_xaxis('', tickmode='linear', dtick=1),
    yaxis=flr_yaxis('Operations', rangemode='tozero'),
)

save_figure(fig_c2, 'figure_c2_pillar_ops', width=1200, height=520)
fig_c2

## Figure D: Timeliness Improvement Slope Chart

Extends the Slide 9 timeliness view back to 2022 (Slide 9 starts at 2023).
Three metrics are tracked: disaster-to-approval, NS-request-to-approval, and days in HQ.
An outlier note is added to the 2026 Days-in-HQ mean (median stays at 1 day).

In [ ]:
# Figure D: Timeliness Improvement Slope Chart
timing = q1_history[q1_history['approval_year'].between(2022, 2026)].copy()
timing['ns_to_approval'] = (
    timing['approval_date'] - timing['date_of_appeal_request_from_ns']
).dt.days

metric_defs = [
    ('Disaster → Approval', 'average_time_disaster_to_approval', IFRC_RED),
    ('NS Request → Approval', 'ns_to_approval', NAVY),
    ('Days in HQ', 'days_in_hq', AMBER),
]

slope_records = []
for label, column, _ in metric_defs:
    grouped = timing.groupby('approval_year')[column].mean().reset_index()
    grouped.columns = ['approval_year', 'mean_days']
    grouped['metric'] = label
    grouped['approval_year'] = grouped['approval_year'].astype(int)
    slope_records.append(grouped)
slope_df = pd.concat(slope_records, ignore_index=True)

fig = go.Figure()
fig.add_shape(
    type='rect',
    x0=2024.8,
    x1=2026.2,
    y0=0,
    y1=float(slope_df[slope_df['metric'] == 'Disaster → Approval']['mean_days'].max()) * 1.1,
    fillcolor='rgba(46, 196, 182, 0.06)',
    line={'width': 0},
    layer='below',
)

for label, column, color in metric_defs:
    subset = slope_df[slope_df['metric'] == label].sort_values('approval_year')
    fig.add_trace(go.Scatter(
        x=subset['approval_year'],
        y=subset['mean_days'],
        mode='lines+markers+text',
        name=label,
        line={'color': color, 'width': 3, 'shape': 'linear'},
        marker={'size': 11, 'color': color, 'line': {'color': FLR_PAPER, 'width': 1.5}},
        text=[f'{value:.1f}d' for value in subset['mean_days']],
        textposition='top center',
        textfont={'color': color, 'size': 14, 'family': FLR_FONT},
    ))

fig.update_layout(
    **flr_layout(
        'Timeliness Improvement — Slope Chart',
        subtitle='Mean processing days — Q1 2022 to 2026',
        height=690,
        margin_b=175,
        margin_t=125,
        margin_r=110,
    ),
    legend=flr_legend(y=-0.18),
    xaxis=flr_xaxis('', tickmode='linear', dtick=1),
    yaxis=flr_yaxis('Mean Days'),
)

save_figure(fig, 'figure_d_timeliness_slope', width=1400, height=690)
fig


## Figure E: Regional Allocation Heatmap

Side-by-side heatmaps — left shows approved CHF, right shows operation count — so a reader
can see whether high-CHF regions have proportionally few large allocations or many small ones.
Rows = regions, columns = Q1 years.
Cell labels show the formatted value so the figure is self-contained without a colour-scale key.

In [9]:
# Figure E: Regional Allocation Heatmap
heat_src = (
    q1_history
    .groupby(['region', 'approval_year'], as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'), ops=('appeal_id', 'nunique'))
)
heat_src['approval_year'] = heat_src['approval_year'].astype(int)

years = sorted(heat_src['approval_year'].unique())
regions = sorted(heat_src['region'].unique())
year_labels = [str(year) for year in years]

pivot_chf = (
    heat_src.pivot(index='region', columns='approval_year', values='total_chf')
    .reindex(index=regions, columns=years)
    .fillna(0)
)
pivot_ops = (
    heat_src.pivot(index='region', columns='approval_year', values='ops')
    .reindex(index=regions, columns=years)
    .fillna(0)
)


def split_text_layers(values_frame, formatter, threshold_ratio=0.45):
    max_value = float(values_frame.to_numpy().max()) if values_frame.to_numpy().size else 1.0
    if max_value <= 0:
        max_value = 1.0

    dark_x, dark_y, dark_text = [], [], []
    light_x, light_y, light_text = [], [], []

    for region in values_frame.index:
        for year in values_frame.columns:
            value = float(values_frame.loc[region, year])
            label = formatter(value)
            if value / max_value < threshold_ratio:
                dark_x.append(str(year))
                dark_y.append(region)
                dark_text.append(label)
            else:
                light_x.append(str(year))
                light_y.append(region)
                light_text.append(label)

    return {
        'dark': {'x': dark_x, 'y': dark_y, 'text': dark_text},
        'light': {'x': light_x, 'y': light_y, 'text': light_text},
    }


chf_layers = split_text_layers(pivot_chf, human_chf, threshold_ratio=0.42)
ops_layers = split_text_layers(pivot_ops, lambda value: str(int(value)), threshold_ratio=0.34)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=['Approved CHF', 'Operations'],
    horizontal_spacing=0.18,
)

fig.add_trace(
    go.Heatmap(
        z=pivot_chf.values,
        x=year_labels,
        y=regions,
        colorscale=[[0, '#fff2df'], [0.35, '#f6c470'], [0.7, '#f08a3c'], [1.0, '#c92a0b']],
        colorbar={
            'x': 0.44,
            'len': 0.82,
            'thickness': 18,
            'title': {'text': 'CHF', 'font': FLR_AXIS_LBL},
            'tickfont': FLR_TICK,
            'outlinewidth': 0,
        },
        hovertemplate='%{y} %{x}: CHF %{z:,.0f}<extra></extra>',
        xgap=4,
        ygap=4,
        showscale=True,
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=pivot_ops.values,
        x=year_labels,
        y=regions,
        colorscale=[[0, '#e7f1fb'], [0.35, '#8bbfe5'], [0.7, '#2f77b5'], [1.0, '#123a73']],
        colorbar={
            'x': 1.03,
            'len': 0.82,
            'thickness': 18,
            'title': {'text': 'Ops', 'font': FLR_AXIS_LBL},
            'tickfont': FLR_TICK,
            'outlinewidth': 0,
        },
        hovertemplate='%{y} %{x}: %{z:.0f} ops<extra></extra>',
        xgap=4,
        ygap=4,
        showscale=True,
        showlegend=False,
    ),
    row=1,
    col=2,
)

for layer_name, color in [('dark', INK), ('light', 'white')]:
    layer = chf_layers[layer_name]
    if layer['text']:
        fig.add_trace(
            go.Scatter(
                x=layer['x'],
                y=layer['y'],
                mode='text',
                text=layer['text'],
                textfont={'size': 14, 'color': color, 'family': FLR_FONT},
                hoverinfo='skip',
                showlegend=False,
            ),
            row=1,
            col=1,
        )

for layer_name, color in [('dark', INK), ('light', 'white')]:
    layer = ops_layers[layer_name]
    if layer['text']:
        fig.add_trace(
            go.Scatter(
                x=layer['x'],
                y=layer['y'],
                mode='text',
                text=layer['text'],
                textfont={'size': 14, 'color': color, 'family': FLR_FONT},
                hoverinfo='skip',
                showlegend=False,
            ),
            row=1,
            col=2,
        )

fig.update_layout(
    **flr_layout(
        'Regional Allocation Heatmap',
        subtitle='CHF and operations — Q1 2022 to 2026',
        height=650,
        margin_l=140,
        margin_r=120,
        margin_t=135,
        margin_b=70,
    ),
)

for column_index in [1, 2]:
    fig.update_xaxes(
        tickfont=FLR_TICK,
        showline=True,
        linecolor=FLR_LINE,
        linewidth=1,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        row=1,
        col=column_index,
    )
    fig.update_yaxes(
        tickfont={'size': 18, 'color': '#444', 'family': FLR_FONT},
        showline=False,
        ticks='',
        row=1,
        col=column_index,
    )

for annotation in fig.layout.annotations:
    if annotation.text in ['Approved CHF', 'Operations']:
        annotation.update(font={'size': 18, 'color': '#444', 'family': FLR_FONT}, yshift=10)

save_figure(fig, 'figure_e_regional_heatmap', width=1500, height=650)
fig

In [38]:
# ── Figure D (Revised): Timeliness — Two-Panel Box Plot ──────────────────────
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning)

ROOT_D = Path('..').resolve()
MASTER_XL = ROOT_D / 'DREF_MasterDataset_v1.1 .xlsx'
OUTPUT_DIR_D = ROOT_D / 'outputs' / 'new_figures'
OUTPUT_DIR_D.mkdir(parents=True, exist_ok=True)

_FLR_PAPER = '#fafafa'
_FLR_FONT  = 'Inter, Segoe UI, Helvetica Neue, Arial, sans-serif'
_FLR_GRID  = '#e8e8e8'
_FLR_LINE  = '#d0d0d0'
_IFRC_RED  = '#E03C31'
_NAVY      = '#2B3A67'
_AMBER     = '#F5A623'
_TEAL      = '#2EC4B6'
_INK       = '#222222'

raw = pd.read_excel(MASTER_XL, sheet_name='ALL_DATA')
mask = ~((raw['Appeal Type'] == 'EA') & (raw['Allocation Type'] == 'Loan'))
df_all = raw[mask & raw['Year'].between(2022, 2026)].copy()
df_all['approval_dt'] = pd.to_datetime(df_all['Date of Approval EnC (start date)'], errors='coerce')
df_q1 = df_all[
    df_all['approval_dt'].dt.month.isin([1, 2, 3]) &
    df_all['approval_dt'].dt.year.between(2022, 2026)
].copy()
df_q1['approval_year'] = df_q1['approval_dt'].dt.year
df_q1['dis_to_appr'] = pd.to_numeric(df_q1['Average time (disaster to approval)'], errors='coerce')
df_q1['ns_to_appr'] = (
    df_q1['approval_dt'] -
    pd.to_datetime(df_q1['Date of Appeal request from NS'], errors='coerce')
).dt.days

resp_q1 = df_q1[df_q1['Pillar'] == 'Response'].copy()
anti_q1  = df_q1[df_q1['Pillar'] == 'Anticipatory'].copy()

resp_disaster_plot = resp_q1[resp_q1['dis_to_appr'].notna() & (resp_q1['dis_to_appr'] >= 0)].copy()
resp_request_plot  = resp_q1[resp_q1['ns_to_appr'].notna()].copy()

anti_q1 = anti_q1[anti_q1['dis_to_appr'].notna()].copy()
anti_q1['days_before_trigger'] = -anti_q1['dis_to_appr']
anti_plot = anti_q1[anti_q1['days_before_trigger'] >= 0].copy()

most_anti     = anti_plot.nlargest(1, 'days_before_trigger').copy()
anti_box_plot = anti_plot.drop(index=most_anti.index).copy()

_YEARS = [2022, 2023, 2024, 2025, 2026]

def iqr_fence_clip(series):
    if len(series) < 4:
        return series
    q1_v = series.quantile(0.25)
    q3_v = series.quantile(0.75)
    iqr  = q3_v - q1_v
    return series[(series >= q1_v - 1.5 * iqr) & (series <= q3_v + 1.5 * iqr)]

resp_disaster_clipped = {
    yr: iqr_fence_clip(resp_disaster_plot.loc[resp_disaster_plot['approval_year'] == yr, 'dis_to_appr'])
    for yr in _YEARS
}
resp_request_clipped = {
    yr: iqr_fence_clip(resp_request_plot.loc[resp_request_plot['approval_year'] == yr, 'ns_to_appr'])
    for yr in _YEARS
}

fastest_resp_per_year = (
    resp_disaster_plot
    .loc[resp_disaster_plot.groupby('approval_year')['dis_to_appr'].idxmin()]
    .reset_index(drop=True)
)

# Fastest anticipatory per year (Chile already excluded via anti_box_plot)
fastest_anti_per_year = (
    anti_box_plot
    .loc[anti_box_plot.groupby('approval_year')['days_before_trigger'].idxmax()]
    .reset_index(drop=True)
) if not anti_box_plot.empty else pd.DataFrame(columns=['approval_year', 'Country', 'days_before_trigger'])

print(f"Q1 Response rows, disaster metric    : {len(resp_disaster_plot)}")
print(f"Q1 Response rows, NS request metric  : {len(resp_request_plot)}")
print(f"Q1 Anticipatory rows (box only)      : {len(anti_box_plot)}")
print(f"\nResponse IQR-clipped per year:")
for yr in _YEARS:
    s = resp_disaster_clipped[yr]
    orig = resp_disaster_plot.loc[resp_disaster_plot['approval_year'] == yr, 'dis_to_appr']
    print(f"  {yr}: {len(s)}/{len(orig)} kept  min={s.min():.0f}  median={s.median():.1f}  max={s.max():.0f}")
print(f"\nFastest per year:\n{fastest_resp_per_year[['approval_year','Country','dis_to_appr']].to_string(index=False)}")
print(f"\nMost anticipatory: {most_anti[['Country','approval_year','days_before_trigger']].to_string(index=False)}")



Q1 Response rows, disaster metric    : 133
Q1 Response rows, NS request metric  : 135
Q1 Anticipatory rows (box only)      : 4

Response IQR-clipped per year:
  2022: 19/22 kept  min=9  median=11.0  max=21
  2023: 20/21 kept  min=0  median=10.5  max=18
  2024: 25/26 kept  min=3  median=14.0  max=23
  2025: 21/23 kept  min=8  median=13.0  max=20
  2026: 39/41 kept  min=2  median=10.0  max=18

Fastest per year:
 approval_year    Country  dis_to_appr
          2022   Zimbabwe          9.0
          2023 Madagascar          0.0
          2024      Chile          3.0
          2025    Burundi          8.0
          2026     Gambia          2.0

Most anticipatory: Country  approval_year  days_before_trigger
  Chile           2025                350.0


In [42]:
YEARS = [2022, 2023, 2024, 2025, 2026]
LEFT_OFFSET = 0.16
RIGHT_OFFSET = 0.16

fig_d2 = make_subplots(
    rows=1, cols=1,
    subplot_titles=['<b>Response Operations</b>'],
)

# Response panel only
for year in YEARS:
    disaster_year = resp_disaster_clipped[year]
    request_year = resp_request_clipped[year]

    if not disaster_year.empty:
        fig_d2.add_trace(go.Box(
            x=np.full(len(disaster_year), year - LEFT_OFFSET),
            y=disaster_year,
            name='Disaster → Approval',
            marker=dict(color=_IFRC_RED, size=5),
            fillcolor='rgba(224,60,49,0.12)',
            line=dict(color=_IFRC_RED, width=2),
            boxmean=True,
            whiskerwidth=0.5,
            boxpoints=False,
            width=0.22,
            legendgroup='resp_disaster',
            showlegend=(year == YEARS[0]),
            hovertemplate='<b>Disaster → Approval</b><br>Year: ' + str(year) + '<br>Days: %{y:.0f}<extra></extra>',
        ), row=1, col=1)

    if not request_year.empty:
        fig_d2.add_trace(go.Box(
            x=np.full(len(request_year), year + RIGHT_OFFSET),
            y=request_year,
            name='NS Request → Approval',
            marker=dict(color=_NAVY, size=5),
            fillcolor='rgba(43,58,103,0.12)',
            line=dict(color=_NAVY, width=2),
            boxmean=True,
            whiskerwidth=0.5,
            boxpoints=False,
            width=0.22,
            legendgroup='resp_request',
            showlegend=(year == YEARS[0]),
            hovertemplate='<b>NS Request → Approval</b><br>Year: ' + str(year) + '<br>Days: %{y:.0f}<extra></extra>',
        ), row=1, col=1)

# Fastest Response stars (one per year)
if not fastest_resp_per_year.empty:
    resp_star_x = fastest_resp_per_year['approval_year'].to_numpy(dtype=float) - LEFT_OFFSET
    resp_text_pos = ['top right' if v == 0 else 'bottom right' for v in fastest_resp_per_year['dis_to_appr']]

    fig_d2.add_trace(go.Scatter(
        x=resp_star_x,
        y=fastest_resp_per_year['dis_to_appr'],
        mode='markers+text',
        name='Fastest approval',
        marker=dict(symbol='star', size=22, color=_TEAL, opacity=0.98, line=dict(color='white', width=1.8)),
        text=[f"{row['Country']} ({row['dis_to_appr']:.0f}d)" for _, row in fastest_resp_per_year.iterrows()],
        textposition=resp_text_pos,
        textfont=dict(color=_TEAL, size=12, family=_FLR_FONT),
        hovertemplate='<b>%{text}</b><extra></extra>',
        legendgroup='stars',
        showlegend=True,
    ), row=1, col=1)

fig_d2.add_hline(
    y=0,
    line_dash='dot',
    line_color='rgba(120,120,120,0.35)',
    line_width=1.2,
    row=1,
    col=1,
)

fig_d2.update_layout(
    title=dict(
        text=(
            '<b>Response Timeliness Distribution — Q1 2022 to 2026</b>'
            '<br><span style="font-size:16px;color:#888;">'
            'line = median, dashed = mean · ★ = fastest approval per year'
            '</span>'
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=22, color='#1a1a2e', family=_FLR_FONT),
        pad=dict(b=20),
    ),
    paper_bgcolor=_FLR_PAPER,
    plot_bgcolor=_FLR_PAPER,
    margin=dict(l=80, r=80, t=145, b=180),
    height=720,
    font=dict(family=_FLR_FONT, size=13, color='#444'),
    legend=dict(
        orientation='h',
        xanchor='center',
        x=0.5,
        yanchor='top',
        y=-0.16,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=13, color='#555', family=_FLR_FONT),
        itemsizing='constant',
    ),
    hoverlabel=dict(bgcolor='white', bordercolor='#ddd', font_size=13, font_family=_FLR_FONT),
)

fig_d2.update_xaxes(
    tickvals=YEARS,
    ticktext=[str(yr) for yr in YEARS],
    tickfont=dict(size=15, color='#555', family=_FLR_FONT),
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor=_FLR_LINE,
    linewidth=1,
    ticks='outside',
    ticklen=6,
    automargin=True,
    range=[2021.55, 2026.45],
    row=1,
    col=1,
)

fig_d2.update_yaxes(
    title=dict(text='Days', font=dict(size=18, color='#333', family=_FLR_FONT), standoff=12),
    tickfont=dict(size=15, color='#555', family=_FLR_FONT),
    gridcolor=_FLR_GRID,
    gridwidth=0.7,
    showgrid=True,
    zeroline=False,
    showline=False,
    ticks='outside',
    ticklen=6,
    automargin=True,
    rangemode='tozero',
    row=1,
    col=1,
)

for ann in fig_d2.layout.annotations:
    if ann.text == '<b>Response Operations</b>':
        ann.update(font=dict(size=16, color='#333', family=_FLR_FONT), yshift=8)

fig_d2.write_image(OUTPUT_DIR_D / 'figure_d2_response_timeliness.png', width=1200, height=720, scale=2)
fig_d2.write_image(OUTPUT_DIR_D / 'figure_d2_response_timeliness.svg', width=1200, height=720)
fig_d2

In [41]:
# Figure D3: Anticipatory Operations (sparse years shown as points)
# Rationale: each year has 1 operation after filtering, so a box plot collapses to a line.

fig_d3 = go.Figure()

# Plot one point per available year (Chile outlier excluded from scale)
if not anti_box_plot.empty:
    fig_d3.add_trace(go.Scatter(
        x=fastest_anti_per_year['approval_year'],
        y=fastest_anti_per_year['days_before_trigger'],
        mode='markers+text',
        name='Fastest anticipatory',
        marker=dict(symbol='star', size=20, color=_TEAL, opacity=0.98, line=dict(color='white', width=1.8)),
        text=[f"{row['Country']} ({row['days_before_trigger']:.0f}d)" for _, row in fastest_anti_per_year.iterrows()],
        textposition='top right',
        textfont=dict(color=_TEAL, size=12, family=_FLR_FONT),
        hovertemplate='<b>%{text}</b><extra></extra>',
    ))

    fig_d3.add_trace(go.Scatter(
        x=anti_box_plot['approval_year'],
        y=anti_box_plot['days_before_trigger'],
        mode='markers',
        name='Operations',
        marker=dict(symbol='circle', size=8, color=_AMBER, opacity=0.85),
        hovertemplate='<b>%{x}</b><br>%{y:.0f} days before trigger<extra></extra>',
        showlegend=False,
    ))

# Chile shown as off-scale callout only
if not most_anti.empty:
    chile_row = most_anti.iloc[0]
    x_chile = int(chile_row['approval_year'])
    d_chile = int(chile_row['days_before_trigger'])

    fig_d3.add_shape(
        type='line',
        x0=x_chile,
        x1=x_chile,
        y0=58,
        y1=63,
        line=dict(color=_TEAL, width=2, dash='dot'),
    )
    fig_d3.add_annotation(
        x=x_chile,
        y=64,
        text=f"{chile_row['Country']} {x_chile}: {d_chile}d early (off-scale)",
        showarrow=False,
        bgcolor='white',
        bordercolor=_TEAL,
        borderwidth=1,
        borderpad=5,
        font=dict(size=11, color=_TEAL, family=_FLR_FONT),
    )

fig_d3.add_hline(
    y=0,
    line_dash='dot',
    line_color='rgba(120,120,120,0.35)',
    line_width=1.2,
    annotation_text='Trigger date',
    annotation_position='right',
    annotation_font=dict(size=11, color='#888', family=_FLR_FONT),
)

fig_d3.update_layout(
    title=dict(
        text=(
            '<b>Anticipatory Timeliness (Sparse Operations) — Q1 2022 to 2026</b>'
            '<br><span style="font-size:16px;color:#888;">'
            'Each point is one operation; box plot removed because n=1 per year in available years'
            '</span>'
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=22, color='#1a1a2e', family=_FLR_FONT),
        pad=dict(b=20),
    ),
    paper_bgcolor=_FLR_PAPER,
    plot_bgcolor=_FLR_PAPER,
    margin=dict(l=90, r=80, t=145, b=130),
    height=650,
    font=dict(family=_FLR_FONT, size=13, color='#444'),
    legend=dict(
        orientation='h',
        xanchor='center',
        x=0.5,
        yanchor='top',
        y=-0.14,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=13, color='#555', family=_FLR_FONT),
    ),
)

fig_d3.update_xaxes(
    tickvals=YEARS,
    ticktext=[str(yr) for yr in YEARS],
    tickfont=dict(size=15, color='#555', family=_FLR_FONT),
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor=_FLR_LINE,
    linewidth=1,
    ticks='outside',
    ticklen=6,
    automargin=True,
    range=[2021.55, 2026.45],
)

fig_d3.update_yaxes(
    title=dict(text='Days Before Trigger', font=dict(size=18, color='#333', family=_FLR_FONT), standoff=12),
    tickfont=dict(size=15, color='#555', family=_FLR_FONT),
    gridcolor=_FLR_GRID,
    gridwidth=0.7,
    showgrid=True,
    zeroline=False,
    showline=False,
    ticks='outside',
    ticklen=6,
    automargin=True,
    range=[-2, 65],
)

fig_d3.write_image(OUTPUT_DIR_D / 'figure_d3_anticipatory_sparse.png', width=1200, height=650, scale=2)
fig_d3.write_image(OUTPUT_DIR_D / 'figure_d3_anticipatory_sparse.svg', width=1200, height=650)
fig_d3